# Protegendo conversas com Amazon Bedrock Guardrails e AgentCore Memory

## Visão Geral

Este tutorial demonstra como integrar o Amazon Bedrock Guardrails com o AgentCore Memory para criar um agente conversacional seguro. Você construirá um agente que filtra conteúdo sensível enquanto mantém o contexto da conversa entre interações.

### Detalhes do Tutorial

| Informação          | Detalhes                                                         |
|:--------------------|:-----------------------------------------------------------------|
| Tipo de tutorial    | Integração Guardrails / Memory                                   |
| Tipo de agente      | Agente com Memory e Guardrails                                   |
| Framework de agentes| Strands Agents                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                      |
| Funcionalidades     | Guardrails, Integração com Memory, Filtragem de Conteúdo        |
| Complexidade        | Intermediário                                                    |
| SDK utilizado       | Amazon Bedrock Python SDK e Bedrock Memory SDK                   |

### O que Você Aprenderá

Neste tutorial, você aprenderá:
1. Como criar um recurso de memória para seu agente
2. Como implementar Amazon Bedrock Guardrails com filtragem de conteúdo
3. Como construir um hook personalizado que combina funcionalidades de guardrails e memória
4. Como armazenar seletivamente o histórico de conversas seguras
5. Como testar seu agente seguro com diferentes tipos de conteúdo

### Arquitetura

Este exemplo demonstra a integração de guardrails com memória para conversas seguras:

<div style="text-align:left">
    <img src="guardrails_memory_flow.png" width="90%"/>
</div>

## 0. Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS configuradas com acesso ao AgentCore Memory e Amazon Bedrock
* Acesso ao modelo Amazon Bedrock (Claude Haiku 4.5)
* Amazon Bedrock Memory SDK

Primeiro, vamos instalar as bibliotecas necessárias:

In [ ]:
!pip install -qr requirements.txt

In [ ]:
# Imports
import os
import boto3
import uuid
import logging
from typing import Dict
from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.memory import MemoryClient, MemorySessionManager
from botocore.exceptions import ClientError
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from strands.experimental.hooks import AfterModelInvocationEvent
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig

# Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("secure-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2') # AWS region for the agent
bedrock_client = boto3.client('bedrock', region_name=REGION)
bedrock_runtime_client = boto3.client('bedrock-runtime', region_name=REGION)
memory_client = MemoryClient(region_name=REGION)

## 1. Criando Amazon Bedrock Guardrails

Nesta seção, criaremos um guardrail para aplicar políticas de segurança de conteúdo ao nosso agente. Guardrails atuam como filtros de segurança que podem ser aplicados tanto às entradas dos usuários quanto às saídas do modelo. Para nosso exemplo, criaremos um guardrail com duas políticas específicas:

1. **Filtragem de Entrada**: Bloquear linguagem ofensiva dos usuários
2. **Filtragem de Saída**: Impedir que o modelo discuta tópicos políticos

Esta abordagem demonstra como guardrails podem proteger contra diferentes tipos de conteúdo problemático em ambas as direções da conversa. O filtro de entrada ajuda a manter um ambiente de conversa respeitoso, enquanto o filtro de saída garante que o modelo não discuta tópicos potencialmente sensíveis.

O objetivo final é evitar salvar mensagens indesejadas em nossa memória, garantindo que apenas conteúdo apropriado seja armazenado para contexto futuro.

In [ ]:
# Unique identifier for this request
unique_id = str(uuid.uuid4())[:6]

# Define guardrail configuration
guardrail_name = f"SecureConversationGuardrail_{unique_id}"
guardrail_description = "Blocks insults in input and political content in output"

try:
    # Create the guardrail
    response = bedrock_client.create_guardrail(
        name=guardrail_name,
        description=guardrail_description,
        # Block insults in input
        contentPolicyConfig={
            'filtersConfig': [
                {
                    'type': 'INSULTS',
                    'inputStrength': 'MEDIUM',
                    'outputStrength': 'MEDIUM',
                    'inputModalities': ['TEXT'],
                    'outputModalities': ['TEXT'],
                    'inputAction': 'BLOCK',
                    'outputAction': 'NONE',
                    'inputEnabled': True,
                    'outputEnabled': False
                }
            ],
            'tierConfig': {
                'tierName': 'CLASSIC'
            }
        },
        # Block political content in output
        topicPolicyConfig={
            'topicsConfig': [
                {
                    'name': 'Politics',
                    'definition': 'Content related to political leaders, elections, political parties, or government affairs',
                    'examples': [
                        'Who is the current president?',
                        'Tell me about the upcoming election',
                        'Explain the political situation in Congress'
                    ],
                    'type': 'DENY',
                    'inputAction': 'NONE',
                    'outputAction': 'BLOCK',
                    'inputEnabled': False,
                    'outputEnabled': True
                }
            ],
            'tierConfig': {
                'tierName': 'CLASSIC'
            }
        },
        blockedInputMessaging="I'm sorry, but your message contains inappropriate language. Please rephrase your question without insults.",
        blockedOutputsMessaging="I apologize, but I cannot provide information on political topics. Is there something else I can help you with?",
    )
    
    # Store guardrail ID for later use
    guardrail_id = response['guardrailId']
    guardrail_arn = response['guardrailArn']
    guardrail_version = "DRAFT"  # New guardrails are created as DRAFT
    
    print(f"✅ Created guardrail: {guardrail_id} (ARN: {guardrail_arn})")
    
except Exception as e:
    print(f"❌ Error creating guardrail: {e}")
    # If the guardrail already exists, try to find its ID
    try:
        response = bedrock_client.list_guardrails()
        existing_guardrail = next((g for g in response['guardrailSummaries'] 
                                  if g['name'] == guardrail_name), None)
        if existing_guardrail:
            guardrail_id = existing_guardrail['guardrailId']
            guardrail_version = "DRAFT"  # Use DRAFT version
            print(f"Using existing guardrail: {guardrail_id}")
    except Exception as list_error:
        print(f"❌ Error listing guardrails: {list_error}")
        guardrail_id = None
        guardrail_version = None


## 2. Criando o Recurso de Memória

Nesta seção, criaremos um recurso de memória para o nosso agente armazenar o histórico de conversas. A memória permite que o agente relembre interações passadas, mantenha o contexto e forneça respostas mais coerentes ao longo do tempo. Ao combinar memória com guardrails, podemos garantir que apenas conteúdo apropriado seja armazenado para referência futura.

Para este exemplo, criaremos um recurso simples de memória de curto prazo sem estratégias adicionais, que é perfeito para manter o contexto da conversa dentro de uma sessão. A memória armazenará mensagens que passaram pelas verificações do nosso guardrail, garantindo que conteúdo inadequado seja filtrado.

In [ ]:
memory_name = f"SecureAgentMemory_{unique_id}"

try:
    # Create memory resource without strategies (thus only access to short-term memory)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # No strategies for short-term memory
        description="Short-term memory for personal agent with guardrails",
        event_expiry_days=7,
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Show any errors during memory creation
    logger.error(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")


## 3. Integrando Bedrock Guardrails, Strands e AgentCore Memory

Nesta seção, criaremos hooks personalizados que integram guardrails com a funcionalidade de memória. Nossa implementação irá:

1. Verificar tanto as entradas dos usuários quanto as saídas do modelo usando Amazon Bedrock Guardrails
2. Substituir conteúdo inadequado por alternativas seguras
3. Armazenar na memória apenas mensagens que passaram pelas verificações do guardrail
4. Recuperar o contexto de conversas anteriores da memória quando o agente for inicializado

Esta abordagem garante que nosso agente mantenha um histórico de conversas limpo enquanto ainda se beneficia das capacidades de memória. Vamos construir os componentes necessários:

In [ ]:
class GuardrailsEvaluator:
    """Reusable guardrails evaluation utility."""
    
    def __init__(self, guardrail_id: str, guardrail_version: str):
        """Initialize the guardrails evaluator.
        
        Args:
            guardrail_id: The ID of the guardrail to use
            guardrail_version: The version of the guardrail (e.g., "DRAFT")
        """
        self.guardrail_id = guardrail_id
        self.guardrail_version = guardrail_version
    
    def evaluate_content(self, content: str, source: str) -> Dict:
        """Evaluate content using Bedrock Guardrails and return result.
        
        Args:
            content: The text content to evaluate
            source: The source type ("INPUT" or "OUTPUT")
            
        Returns:
            Dict containing guardrail evaluation results
        """
        try:
            logger.info(f"⏳ CHECKING {source}: '{content[:30]}...'")
            
            response = bedrock_runtime_client.apply_guardrail(
                guardrailIdentifier=self.guardrail_id,
                guardrailVersion=self.guardrail_version,
                source=source,
                content=[{"text": {"text": content}}]
            )
            
            action = response.get('action')
            logger.info(f"🔍 GUARDRAIL ACTION: {action}")
            
            return response
        except Exception as e:
            logger.error(f"❌ Guardrail evaluation failed: {e}")
            return {"error": str(e)}


class GuardrailsHookProvider(HookProvider):
    """Hook provider that combines guardrails enforcement with memory storage."""
    
    def __init__(self, guardrails_evaluator: GuardrailsEvaluator):
        self.evaluator = guardrails_evaluator
        self.blocked_outputs = set()

    def after_model_invocation(self, event: AfterModelInvocationEvent) -> None:
        """Check model output with guardrails and replace if needed.
        
        Args:
            event: Event containing the model response
        """
        # Skip if model invocation failed
        if event.exception is not None or event.stop_response is None:
            logger.error("⚠️ Model invocation failed, skipping guardrail check")
            return
        
        logger.info("🔍 AfterModelInvocationEvent: Checking model output")
        
        # Extract message from the model response
        message = event.stop_response.message
        
        # Extract content
        if isinstance(message.get("content"), list):
            content = "".join(block.get("text", "") for block in message.get("content", []))
        else:
            content = str(message.get("content", ""))
        
        content_id = hash(content)
        
        # Check against guardrails
        result = self.evaluator.evaluate_content(content, "OUTPUT")
        
        # Handle guardrail violations
        if result.get("action") == "GUARDRAIL_INTERVENED":
            logger.warning("⛔ ASSISTANT MESSAGE BLOCKED BY GUARDRAILS")
            
            # Mark this output as blocked
            self.blocked_outputs.add(content_id)
            
            # Get the guardrail-provided alternative if available
            replacement_content = None
            if "outputs" in result and result["outputs"] and len(result["outputs"]) > 0:
                if "text" in result["outputs"][0]:
                    replacement_content = result["outputs"][0]["text"]
            
            # Fall back to generic message if no replacement provided
            if not replacement_content:
                replacement_content = "I apologize, but I cannot provide the requested information as it would violate our content policies."
            
            # Update the message content - THIS WILL CHANGE WHAT THE USER SEES
            if isinstance(message.get("content"), list):
                message["content"] = [{"text": replacement_content}]
            else:
                message["content"] = replacement_content
            
            logger.info(f"⚠️ Replaced assistant message with guardrail response: {replacement_content[:30]}...")
    
    
    def register_hooks(self, registry: HookRegistry):
        """Register all hooks with the registry.
        
        Args:
            registry: The hook registry to register with
        """
        registry.add_callback(AfterModelInvocationEvent, self.after_model_invocation)

## 4. Criando e Configurando o Agente

Nesta seção, criaremos nosso agente conversacional seguro combinando todos os componentes que construímos: o modelo Bedrock, o avaliador de guardrails e o hook provider com memória habilitada. Esta integração cria um agente completo que pode manter conversas enquanto aplica políticas de conteúdo e armazena o contexto apropriado.

In [ ]:
ACTOR_ID = "user_1"
SESSION_ID = "session_001"
#bedrock_model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")

evaluator = GuardrailsEvaluator(
    guardrail_id=guardrail_id,
    guardrail_version=guardrail_version
)

session_manager = None

def create_personal_agent():
    """Create personal agent with memory and guardrails"""
    global session_manager
     # Close previous session manager if it exists
    if session_manager is not None:
        session_manager.close()

    # Configure AgentCore Memory
    config = AgentCoreMemoryConfig(
        memory_id=memory_id,
        session_id=SESSION_ID,
        actor_id=ACTOR_ID
    )

    # Create session manager (explicit lifecycle — closed in cleanup cell)
    session_manager = AgentCoreMemorySessionManager(
        agentcore_memory_config=config,
        region_name=REGION
    )

    # Create agent with session manager and guardrails hook
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
        system_prompt="You are a helpful personal assistant. Be friendly and professional.",
        session_manager=session_manager,
        hooks=[GuardrailsHookProvider(evaluator)],
        callback_handler=None
    )
    return agent
# Create agent
agent = create_personal_agent()
logger.info("✅ Personal agent created with memory and guardrails")

Esta implementação cria um agente seguro que irá:

1. Carregar o contexto de conversa existente da memória quando inicializado
2. Verificar as entradas dos usuários contra guardrails antes do processamento
3. Verificar as saídas do modelo contra guardrails antes de mostrar ao usuário
4. Armazenar apenas mensagens aprovadas na memória para contexto futuro
5. Manter o histórico de conversas entre múltiplas interações

A combinação de guardrails e memória garante que nosso agente mantenha uma experiência de conversa segura e contextual.

## 5. Testando o Agente Seguro

Vamos testar nosso agente com diferentes tipos de entrada para ver como a integração de guardrails e memória funciona na prática. Tentaremos tanto entradas aceitáveis quanto aquelas que podem acionar intervenções do guardrail para verificar se nossa implementação está funcionando corretamente.

Primeiro, vamos criar uma função auxiliar para lidar com a verificação do guardrail e a invocação do agente:

In [ ]:
def process_with_guardrails(user_input):
    """Process user input with guardrails before sending to agent.
    
    Args:
        user_input: The text input from the user
        
    Returns:
        The agent response or guardrail rejection
    """
    # Check input against guardrails
    result = evaluator.evaluate_content(user_input, "INPUT")
    
    if result.get("action") == "GUARDRAIL_INTERVENED":
        # Get rejection message from guardrail
        if "outputs" in result and result["outputs"] and "text" in result["outputs"][0]:
            rejection_content = result["outputs"][0]["text"]
        else:
            rejection_content = "I cannot process that request."
        
        # Return rejection without calling agent
        print(rejection_content)
        return rejection_content
    else:
        # Input passed guardrails, proceed with agent call
        response = agent(user_input)
        print(response)
        return response

### Teste 1: Conversa Normal

Vamos começar com uma saudação normal que deve passar por todos os guardrails:

In [ ]:
print("Test 1: Normal greeting")
user_input = "I am dani."
process_with_guardrails(user_input)

### Teste 2: Conteúdo Ofensivo (Deve Acionar o Guardrail de Entrada)

Vamos tentar uma entrada com linguagem ofensiva que deve ser bloqueada pelo guardrail de entrada:

In [ ]:
print("\nTest 2: Insulting content (should trigger input guardrail)")
user_input = "You're a stupid assistant."
process_with_guardrails(user_input)

### Teste 3: Conteúdo Político (Deve Acionar o Guardrail de Saída)

Agora vamos tentar uma pergunta sobre política, que deve passar pelo guardrail de entrada mas acionar o guardrail de saída:

In [ ]:
print("\nTest 3: Political question (should trigger output guardrail)")
user_input = "Who is the president of the US?"
process_with_guardrails(user_input)

### Examinando o Conteúdo da Memória

Vamos verificar o que foi armazenado na memória após nossos testes:

In [ ]:
# Check what's stored in memory
print("\n=== Memory Contents ===")
manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
session = manager.create_memory_session(actor_id=ACTOR_ID, session_id=SESSION_ID)
recent_turns = session.get_last_k_turns(k=5)


for i, turn in enumerate(recent_turns):
    print(f"\nTurn {i+1}:")
    for msg in turn:
        role = msg['role']
        content = msg['content']['text']
        print(f"- {role}: {content[:100]}...")

### Teste 4: Pergunta de Acompanhamento para Testar a Memória
Vamos fazer uma pergunta de acompanhamento para ver se o agente lembra do contexto anterior:

In [ ]:
agent = create_personal_agent()
print("\nTest 4: Follow-up to test memory")
user_input = "What's my name?"
process_with_guardrails(user_input)

## 6. Limpeza (Opcional)

Quando terminar de experimentar com seu agente seguro, você pode querer limpar os recursos criados neste tutorial. Esta seção mostra como excluir os recursos de guardrail e memória.

In [ ]:
# Close the session manager to flush any buffered messages
if session_manager is not None:
    session_manager.close()
    print("✅ Closed session manager")

# Delete the memory resource
try:
    memory_client.delete_memory_and_wait(memory_id=memory_id)
    print(f"✅ Deleted memory resource: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")

# Delete the guardrail
try:
    bedrock_client.delete_guardrail(
        guardrailIdentifier=guardrail_id
    )
    print(f"✅ Deleted guardrail: {guardrail_id}")
except Exception as e:
    print(f"❌ Error deleting guardrail: {e}")

## Conclusão

Neste tutorial, construímos um agente conversacional seguro que combina Amazon Bedrock Guardrails com capacidades do AgentCore Memory. Nossa implementação:

1. Filtra entradas inadequadas dos usuários usando guardrails
2. Impede que o agente discuta tópicos sensíveis
3. Armazena apenas mensagens aprovadas na memória
4. Mantém o contexto da conversa usando memória para uma experiência de usuário aprimorada

Ao integrar guardrails com memória, você pode construir agentes robustos que mantêm conformidade com políticas de conteúdo enquanto ainda fornecem respostas personalizadas e contextuais. Este padrão pode ser estendido para cenários mais complexos adicionando filtros de guardrail adicionais ou implementando estratégias de memória de longo prazo.